# Univariate EDA: Distributions, Outliers, and Review Notes

This notebook supports the W3-EDA-Gold milestone by focusing on variables marked `conditional` or `needs_group_review` in `data/references/leakage_conceptual_screening.csv`.

The analysis generates exploratory analysis, identifies patterns/anomalies/caveats, saves reusable figures and summary tables, and connects findings to hypotheses and modeling choices.

**Assumptions / limitations:**
- Dataset: `DataCoSupplyChainDataset.csv` in `data/bronze/dataco/`
- Analysis is reproducible and stored in the repository.
- Findings inform modeling decisions; review by team member required.
- Any assumptions, decisions, or limitations are documented.

In [ ]:
import os
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

ROOT = Path.cwd()
FIG_DIR = ROOT / 'notebooks' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_PATH = ROOT / 'notebooks' / 'eda_univariate_summary.csv'

In [ ]:
reference_dir = ROOT / 'data' / 'references'
leakage_file = reference_dir / 'leakage_conceptual_screening.csv'
schema_file = reference_dir / 'silver_schema_data_dictionary.csv'

leakage_df = pd.read_csv(leakage_file)
schema_df = pd.read_csv(schema_file)

review_vars = sorted(leakage_df.loc[leakage_df['screening_status'].isin(['needs_group_review', 'conditional_review']), 'variable_name'].unique())
review_schema = schema_df[schema_df['silver_column_name'].isin(review_vars)].copy()

print(f'Review variables identified: {len(review_vars)}')
review_schema[['silver_column_name', 'silver_data_type', 'review_status', 'leakage_restriction']].head(25)

In [ ]:
dataset_file = ROOT / 'data' / 'bronze' / 'dataco' / 'DataCoSupplyChainDataset.csv'

if not dataset_file.exists():
    raise FileNotFoundError(f'Dataset not found at {dataset_file}. Please ensure DataCoSupplyChainDataset.csv is placed in data/bronze/dataco/')

print('Using dataset file:', dataset_file)
df = pd.read_csv(dataset_file, low_memory=False)
print('Loaded rows:', len(df), 'columns:', len(df.columns))

In [ ]:
def normalize_column_name(name):
    text = str(name).strip().lower()
    text = re.sub(r'[\s_-]+', ' ', text)
    return text

column_lookup = {normalize_column_name(col): col for col in df.columns}

def resolve_column(raw_name):
    normalized = normalize_column_name(raw_name)
    return column_lookup.get(normalized)

def numeric_outlier_stats(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outlier_mask = series.lt(lower) | series.gt(upper)
    return {
        'q1': q1,
        'q3': q3,
        'iqr': iqr,
        'lower_fence': lower,
        'upper_fence': upper,
        'outlier_count': int(outlier_mask.sum()),
        'outlier_rate': float(outlier_mask.mean())
    }

summary_rows = []

for variable_name in review_vars:
    dataset_col = resolve_column(variable_name)
    if dataset_col is None:
        summary_rows.append({
            'variable_name': variable_name,
            'dataset_column': None,
            'status': 'missing_in_dataset',
            'missing_rate': None,
            'unique_values': None,
            'outlier_count': None,
            'notes': 'Column from the review list was not present in the loaded dataset.',
            'decision': 'needs_group_review'
        })
        continue

    series = df[dataset_col] 
    missing_rate = float(series.isna().mean())
    unique_values = int(series.nunique(dropna=True))
    data_type = series.dtype
    notes = []
    figure_name = dataset_col.replace(' ', '_').replace('/', '_').replace('\\\'', '').replace('\\\"', '')
    figure_base = FIG_DIR / f'{figure_name}'
    decision = 'needs_group_review'  # default

    if pd.api.types.is_numeric_dtype(series):
        numeric_series = pd.to_numeric(series, errors='coerce')
        stats = numeric_series.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).to_dict()
        outlier_info = numeric_outlier_stats(numeric_series.dropna())
        notes.append(f'Numeric field with {unique_values} distinct values.')
        notes.append(f'Outlier rate by IQR: {outlier_info["outlier_rate"]:.2%} ({outlier_info["outlier_count"]} outliers).')
        if outlier_info['outlier_rate'] > 0.05:
            notes.append('High outlier rate; review for data quality or transformation.')
            decision = 'needs_group_review'
        else:
            decision = 'approved_for_gold'

        fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(10, 4), constrained_layout=True)
        sns.histplot(numeric_series.dropna(), ax=axes[0], bins=40, color='tab:blue')
        axes[0].set_title(f'Histogram: {dataset_col}')
        sns.boxplot(x=numeric_series.dropna(), ax=axes[1], color='tab:orange')
        axes[1].set_title(f'Boxplot: {dataset_col}')
        fig.savefig(figure_base.with_suffix('.png'))
        plt.close(fig)

        summary_rows.append({
            'variable_name': variable_name,
            'dataset_column': dataset_col,
            'status': 'numeric',
            'missing_rate': missing_rate,
            'unique_values': unique_values,
            'outlier_count': outlier_info['outlier_count'],
            'notes': ' '.join(notes),
            'decision': decision,
            'summary_min': stats.get('min'),
            'summary_q1': stats.get('25%'),
            'summary_median': stats.get('50%'),
            'summary_q3': stats.get('75%'),
            'summary_max': stats.get('max'),
        })
    else:
        top_categories = series.fillna('NULL').astype(str).value_counts(dropna=False).head(20)
        cardinality = unique_values
        notes.append(f'Categorical field with cardinality {cardinality}.')
        if cardinality > 100:
            notes.append('High cardinality; consider grouping or encoding strategy.')
            decision = 'needs_group_review'
        elif missing_rate > 0.1:
            notes.append('High missingness; review imputation or exclusion.')
            decision = 'needs_group_review'
        else:
            decision = 'approved_for_gold'

        fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
        sns.barplot(x=top_categories.values, y=top_categories.index, ax=ax, palette='crest')
        ax.set_title(f'Top categories: {dataset_col}')
        ax.set_xlabel('Count')
        fig.savefig(figure_base.with_suffix('.png'))
        plt.close(fig)

        summary_rows.append({
            'variable_name': variable_name,
            'dataset_column': dataset_col,
            'status': 'categorical',
            'missing_rate': missing_rate,
            'unique_values': cardinality,
            'outlier_count': None,
            'notes': ' '.join(notes),
            'decision': decision,
            'summary_min': None,
            'summary_q1': None,
            'summary_median': None,
            'summary_q3': None,
            'summary_max': None,
        })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(SUMMARY_PATH, index=False)
print('EDA summary saved to', SUMMARY_PATH)
print('Figures saved to', FIG_DIR)
summary_df.head(25)

## Findings Summary

- **Patterns:** [Summarize key patterns observed, e.g., skewed distributions, high cardinality]
- **Anomalies:** [Note any data quality issues, extreme outliers]
- **Caveats:** [Document limitations, e.g., missing data, assumptions]
- **Modeling Connections:** [Link to hypotheses, e.g., high outliers may affect AO2 profitability models]
- **Decisions:** Variables marked 'approved_for_gold' can proceed; 'needs_group_review' require discussion.

**Next Steps:** Review summary table and figures; discuss in team meeting.